<a href="https://colab.research.google.com/github/ibrahimkarym/Project_Restarted_Hybrid-/blob/main/rhp_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Restarted hybridplus (RHP) vs DF-SANE, DPRP+ and GN-BFGS on regularised logistic regression

One notebook containing everything: the problem, the **RHP** algorithm (identity scaling, finite-difference initial step),
**DF-SANE**, **DPRP+**, **GN-BFGS** (derivative-free methods from the papers in your folder), the sanity checks, the regression tests, and the experiment.
The code in the cells is exactly the code of the separate `.py` files that were tested
(`problem.py`, `rhp.py`, `dfsane.py`, `dprpplus.py`, `gnbfgs.py`, `check_gradient.py`, `check_dfsane.py`, `test_edge_cases.py`, `run_experiment.py`).

**Run order:** just *Run All* (Runtime → Run all). At the end the notebook regenerates the LaTeX report (`temp/report.tex` with its figures) from the new results, zips the `temp` folder and starts the download of `temp.zip`. The first code cell downloads the LIBSVM files
(`a1a.t … a9a.t`, `colon-cancer.bz2`) from the LIBSVM site into `LIB/`; if you already have them, upload them
into a `LIB/` folder instead (or set `LIBSVM_DIR`). The full experiment takes roughly 10-15 minutes
(GN-BFGS on colon-cancer alone is ~30 s per run).
Requirements: Python 3, numpy, scipy, scikit-learn.

## Parameters
* **RHP:** `t = 0.8`, `rho = 0.85`, `sigma = 1e-4`, `lmax = 10` (chosen by the user after preliminary runs, not tuned on held-out data); `eta_k = (k+1)^-2`, `lmin = 1e-3`, `nu = 1e-6`.
  `sigma` and `eta_k` of the RHP paper's Remark 3.1 are used as quoted there; **`lmin`, `nu` (and the values of `t`, `rho`, `lmax` above) are choices** not taken from the text of the paper I was given.
* **DF-SANE** (La Cruz et al. 2006, Section 4): `M = 10`, `gamma = 1e-4`, `tau_min = 0.1`, `tau_max = 0.5`, `sigma_min = 1e-10`, `sigma_max = 1e10`,
  `sigma_0 = 1`, `eta_k = ||F(x_0)||/(1+k)^2`, paper's replacement rule for `sigma`. (A `variant="scipy"` exists only to validate the code
  against `scipy.optimize.root(method='df-sane')`: identical in 50/50 runs.)
* **DPRP+** (Mohammad & Awwal 2022): `theta = 1`, `rho = 0.2`, `tau_k = 1/e^((k+1)^(k+1))` from the paper (printed grouping is ambiguous; this is my reading; see the docstring). **`sigma = 1e-4`, `l`, `u` are not given in the paper**:
  two settings are run, `DPRP+` (`l=0.01, u=100`) and `DPRP+(.5,1)` (`l=0.5, u=1`, chosen after a small trial on one start point of a1a.t / colon-cancer, so it is favourable to DPRP+).
* **GN-BFGS** (Li & Fukushima 1999): `r = 0.1`, `rho = sqrt(0.9)`, `sigma1 = sigma2 = 1e-5`, `lambda_{-1} = 0.01`, `eps_k = k^-2` (as `(k+1)^-2`), `B_0 = I`, dense `B_k`.
* **System configuration** (CPU model, cores, RAM, OS, library versions, BLAS threads, Colab or local) is read from the machine at run time and written into `results.json` and into the report.
* **All methods:** stop at `||F(x)|| <= 1e-6` or 2000 iterations; regularisation weight 0.1.

## How to read the numbers
* **Itr** iterations.
* **NF** evaluations of F (GN-BFGS counts every `g` evaluation).
* **CPU** process CPU time in seconds (`time.process_time`), data loading excluded; the wall-clock time (`time.perf_counter`) is stored as well.
* Starting points `4*(rand(n)-0.5)`, numpy generator re-seeded with 0 **for every data set**, so data sets with the same
  `n` share the same five starting vectors; within a data set all methods share them. numpy's stream is not MATLAB's.

In [ ]:
import os, subprocess

# Folder with the LIBSVM files.  Works on Colab (downloads them) and locally (set LIBSVM_DIR to your folder).
DATA_DIR = os.environ.get("LIBSVM_DIR", "LIB")
os.makedirs(DATA_DIR, exist_ok=True)
BASE = "https://www.csie.ntu.edu.tw/~cjlin/libsvmtools/datasets/binary/"
FILES = [f"a{i}a.t" for i in range(1, 10)] + ["colon-cancer.bz2"]
for name in FILES:
    stem = name.replace(".bz2", "")
    have = any(os.path.exists(os.path.join(DATA_DIR, stem + ext)) for ext in ("", ".txt", ".bz2"))
    if not have:
        print("downloading", name)
        subprocess.run(["wget", "-q", "-O", os.path.join(DATA_DIR, name), BASE + name], check=True)
print(sorted(os.listdir(DATA_DIR)))

['a1a.t', 'a2a.t', 'a3a.t', 'a4a.t', 'a5a.t', 'a6a.t', 'a7a.t', 'a8a.t', 'a9a.t', 'colon-cancer.bz2']


## 1. The problem

```
Regularised logistic regression, written as a nonlinear equation F(x) = 0.

    f(x) = (1/N) * sum_i log(1 + exp(-b_i * a_i^T x))  +  (reg/2) * ||x||^2
    F(x) = grad f(x) = (1/N) * sum_i  -b_i * exp(-b_i a_i^T x) / (1 + exp(-b_i a_i^T x)) * a_i  +  reg * x

(eq. (35)-(36) of the "inertial derivative-free projection" paper, with the
regularisation weight written `reg` here; the paper's value is 0.1.)

f is strongly convex, so F has exactly one root.  The Jacobian of F is the
Hessian of f, which is symmetric positive definite.
```

In [ ]:
import os

import numpy as np
from scipy.special import expit
from sklearn.datasets import load_svmlight_file

REG = 0.1


class LogisticProblem:
    def __init__(self, A, b, reg=REG):
        self.A = A.tocsr()          # N x n data matrix (rows are the a_i), sparse
        self.b = b                  # N labels in {-1, +1}
        self.N, self.n = A.shape
        self.reg = reg

    def f(self, x):
        """Objective value f(x)."""
        z = self.b * (self.A @ x)                       # z_i = b_i a_i^T x
        return np.mean(np.logaddexp(0.0, -z)) + 0.5 * self.reg * (x @ x)

    def F(self, x):
        """F(x) = grad f(x)."""
        z = self.b * (self.A @ x)
        w = -self.b * expit(-z)                         # -b_i e^{-z_i} / (1 + e^{-z_i})
        return (self.A.T @ w) / self.N + self.reg * x


# Number of variables n and number of rows N of each data set, as in Table 3 of the paper.
DATASETS = {
    "a1a.t": (30956, 123), "a2a.t": (30296, 123), "a3a.t": (29376, 123),
    "a4a.t": (27780, 123), "a5a.t": (26147, 123), "a6a.t": (21341, 123),
    "a7a.t": (16461, 123), "a8a.t": (9865, 122),  "a9a.t": (16281, 122),
    "colon-cancer": (62, 2000),
}


def load_libsvm(data_dir, name):
    """Load one LIBSVM file.  Accepts the plain name, a .txt suffix, or a .bz2 file."""
    N, n = DATASETS[name]
    for suffix in ("", ".txt", ".bz2"):
        path = os.path.join(data_dir, name + suffix)
        if os.path.isfile(path):
            A, y = load_svmlight_file(path, n_features=n)   # reads .bz2 directly
            b = np.where(y > 0, 1.0, -1.0)                   # labels -> {-1, +1}
            assert A.shape == (N, n), f"{name}: got {A.shape}, expected {(N, n)}"
            return A, b
    raise FileNotFoundError(f"{name} (.txt/.bz2) not found in {data_dir}")

## 2. Restarted hybridplus (RHP)

```
Restarted hybridplus (identity scaling, B_k = I): Algorithm 1 of the paper, derivative-free,
for F(x) = 0.  Equation numbers in the comments refer to that paper.

Merit function   f(x) = 0.5 * ||F(x)||^2.

Direction (k >= 1), with s_{k-1} = x_k - x_{k-1}, y_{k-1} = F_k - F_{k-1}:
    w_{k-1}  = y_{k-1} - (d_{k-1}^T y_{k-1} / ||d_{k-1}||^2) d_{k-1}                      (6)
    z_{k-1}  = d_{k-1} + sqrt(1-t^2) ||d_{k-1}|| / (t ||w_{k-1}||) * w_{k-1}              (7)
    beta_k   = max{0, F_k^T z_{k-1}} / max{d_{k-1}^T z_{k-1}, ||F_{k-1}||^2}               (7)
    d_k      = -F_k + beta_k d_{k-1};   d_0 = -F_0                                         (9)
Restart (d_k = -F_k) if d_{k-1} = 0, or w_{k-1} = 0, or the candidate d_k = 0.            (8)

Line search (two-sided, nonmonotone, derivative free):  starting from lambda = lambda0_k, try
x_k + lambda d_k, then x_k - lambda d_k; accept the first that satisfies
    f(x_k +- lambda d_k) <= (1 + eta_k) f_k - sigma lambda^2 ||d_k||^2                     (10)
otherwise lambda <- rho * lambda.   eta_k = (k+1)^-2 (summable, as required by (3)).

Initial trial step  lambda0_k = min{lmax, max{lmin, lambda_hat_k}}                         (11)
with the finite-difference estimate
    lambda_hat_k = |F_k^T d_k| / | d_k^T (F(x_k + nu d_k) - F_k) / nu |
(one extra evaluation of F per iteration; used for k = 0 as well), and lambda_hat := 1 whenever
the estimate is zero, not finite, or undefined.

Following Remark 3.2(ii), the memory term uses the direction d_{k-1}, not the signed
displacement iota * lambda * d_{k-1}.

`nf` in the result counts every evaluation of F (including the extra finite-difference one).
```

In [ ]:
import numpy as np


def rhp(F, x0, t=0.8, rho=0.85, sigma=1e-4, lmin=1e-3, lmax=10.0,
        nu=1e-6, tol=1e-6, maxit=2000, max_backtracks=200, trace=None):
    nf = 0

    def Feval(x):
        nonlocal nf
        nf += 1
        return F(x)

    x = np.array(x0, dtype=float)
    Fk = Feval(x)
    fk = 0.5 * (Fk @ Fk)
    d = -Fk                                  # d_0 = -F_0
    d_prev = F_prev = s_prev = y_prev = None
    restarts = 0
    status = "maxit"
    k = 0

    while k < maxit:
        if trace is not None:
            trace.append(float(np.linalg.norm(Fk)))   # residual of the current iterate (diagnostic only)
        if np.linalg.norm(Fk) <= tol:
            break

        # ---------------- search direction, eqs. (6)-(9) ----------------
        if k >= 1:
            restart = False
            dd_prev = d_prev @ d_prev
            if dd_prev == 0.0:
                restart = True                                   # d_{k-1} = 0
            else:
                w = y_prev - (d_prev @ y_prev) / dd_prev * d_prev                        # (6)
                nw = np.linalg.norm(w)
                if nw == 0.0:
                    restart = True                               # w_{k-1} = 0
                else:
                    z = d_prev + np.sqrt(1.0 - t * t) * np.linalg.norm(d_prev) / (t * nw) * w   # (7)
                    beta = max(0.0, Fk @ z) / max(d_prev @ z, F_prev @ F_prev)           # (7)
                    d = -Fk + beta * d_prev                                              # (9)
                    if not np.any(d):
                        restart = True                           # candidate direction = 0
            if restart:
                d = -Fk
                restarts += 1

        # ---------------- initial trial step, eq. (11) ----------------
        lam_hat = 1.0
        F_shift = Feval(x + nu * d)                              # the one extra evaluation of F
        den = abs(d @ (F_shift - Fk) / nu)
        if den > 0.0:
            v = abs(Fk @ d) / den
            if v > 0.0 and np.isfinite(v):
                lam_hat = v
        lam = min(lmax, max(lmin, lam_hat))

        # ---------------- two-sided nonmonotone line search, eq. (10) ----------------
        eta = (k + 1) ** -2.0
        dd = d @ d
        accepted = False
        for _ in range(max_backtracks):
            for iota in (1.0, -1.0):                             # + trial first, then - trial
                x_new = x + iota * lam * d
                F_new = Feval(x_new)
                f_new = 0.5 * (F_new @ F_new)
                if f_new <= (1.0 + eta) * fk - sigma * lam ** 2 * dd:
                    accepted = True
                    break
            if accepted:
                break
            lam *= rho
        if not accepted:
            status = "line_search_failed"
            break

        # ---------------- update ----------------
        s_prev = x_new - x                                       # s_k = x_{k+1} - x_k
        y_prev = F_new - Fk                                      # y_k = F_{k+1} - F_k
        F_prev, d_prev = Fk, d
        x, Fk, fk = x_new, F_new, f_new
        k += 1

    # convergence is decided from the final stored residual, whichever way the loop ended
    if status == "maxit" and np.linalg.norm(Fk) <= tol:
        status = "converged"

    return dict(x=x, itr=k, nf=nf, normF=float(np.linalg.norm(Fk)), restarts=restarts, status=status)

## 3. DF-SANE

```
DF-SANE: derivative-free spectral residual method for F(x) = 0,
W. La Cruz, J. M. Martinez, M. Raydan, "Spectral residual method without gradient information
for solving large-scale nonlinear systems of equations", Math. Comp. 75 (2006) 1429-1448.

Merit function   f(x) = ||F(x)||^2.

Iteration k:
    d_k = -sigma_k F_k                         (spectral residual direction; sigma_k may be negative)
    Nonmonotone, derivative-free line search: with alpha_+ = alpha_- = 1, try  x_k + alpha_+ d_k  and
    x_k - alpha_- d_k;  a trial point is accepted when
        f(x_k +- alpha d_k) <= fbar_k + eta_k - gamma * alpha^2 * f(x_k),
        fbar_k = max{ f(x_{k-j}) : 0 <= j < min(k+1, M) }        (nonmonotone reference value)
    otherwise alpha_+ and alpha_- are reduced by safeguarded quadratic interpolation
        alpha_new = clip( alpha^2 f_k / (f_trial + (2 alpha - 1) f_k),  tau_min alpha,  tau_max alpha ).
    Spectral coefficient:   sigma_{k+1} = s_k^T s_k / s_k^T y_k,   s_k = x_{k+1} - x_k,  y_k = F_{k+1} - F_k,
    safeguarded to  sigma_eps <= |sigma| <= 1/sigma_eps   (clipped, sign kept).   sigma_0 = 1.

Parameters (Section 4 of the 2006 paper):
    M = 10, gamma = 1e-4, tau_min = 0.1, tau_max = 0.5, sigma_min = 1e-10, sigma_max = 1e10, sigma_0 = 1,
    nexp = 2 (merit f = ||F||^2),  eta_k = ||F(x_0)|| / (1 + k)^2   (summable).
    Spectral coefficient sigma_k = s'_k s_k / s'_k y_k.  If |sigma_k| is outside [sigma_min, sigma_max] the paper
    replaces it by   1 if ||F_k|| > 1;   1/||F_k|| if 1e-5 <= ||F_k|| <= 1;   1e-5 if ||F_k|| < 1e-5
    (the displayed formula in the paper prints "10^5" for the last case but the explanation below it says
    "the fixed value 10^-5"; I follow the explanation).  This replacement "rarely occurs" (paper).

variant="scipy" instead uses SciPy's choices (eta_k = f(x_0)/(1+k)^2 and clipping of sigma to the interval);
it exists only so that check_dfsane.py can verify this code against scipy.optimize.root(method='df-sane').
The default variant="paper" is the one used in the experiment.

Rounding: the merit value is computed as np.linalg.norm(F)**2, exactly as SciPy does.  Computing it as
F @ F is the same number in exact arithmetic but rounds differently, and DF-SANE amplifies a 1e-16
difference into a different trajectory on the colon-cancer problem (one run in five then needed 393
instead of 430 iterations; both converge).  Using SciPy's formula makes the two codes reproduce each other.

If s'y = 0 the quotient s's/s'y is undefined; it is set to inf (out of range), so the variant's safeguard chooses the
next sigma (paper: 1 / 1/||F|| / 1e-5).  Regression test 5 in test_edge_cases.py.

`nf` counts every evaluation of F, including the one at x_0.  Stopping: ||F(x_k)|| <= tol.
`status` is "converged" iff the final stored ||F|| <= tol.
```

In [ ]:
import numpy as np


def dfsane(F, x0, tol=1e-6, maxit=2000, M=10, gamma=1e-4, tau_min=0.1, tau_max=0.5,
           sigma_eps=1e-10, sigma0=1.0, max_backtracks=100, variant="paper", trace=None):
    assert variant in ("paper", "scipy")
    nf = 0

    def Feval(x):
        nonlocal nf
        nf += 1
        return F(x)

    x = np.array(x0, dtype=float)
    Fk = Feval(x)
    fk = np.linalg.norm(Fk) ** 2         # merit ||F||^2 (same rounding as SciPy; F @ F is equal in exact arithmetic)
    f0 = fk if variant == "scipy" else np.sqrt(fk)      # eta_k = (f(x0) or ||F(x0)||)/(1+k)^2
    history = [fk]                       # last M merit values, newest last
    sigma = sigma0
    status = "maxit"
    k = 0

    while k < maxit:
        if trace is not None:
            trace.append(float(np.sqrt(fk)))   # residual of the current iterate (diagnostic only)
        if np.sqrt(fk) <= tol:
            break

        # safeguard of the spectral coefficient (only if |sigma| is outside [sigma_eps, 1/sigma_eps])
        if not (sigma_eps <= abs(sigma) <= 1.0 / sigma_eps):
            if variant == "scipy":
                sigma = np.sign(sigma) / sigma_eps if abs(sigma) > 1.0 / sigma_eps else sigma_eps
            else:                                       # replacement rule of the 2006 paper
                nF = np.sqrt(fk)
                sigma = 1.0 if nF > 1.0 else (1.0 / nF if nF >= 1e-5 else 1e-5)

        d = -sigma * Fk
        eta = f0 / (1.0 + k) ** 2
        fbar = max(history)

        # ---------------- nonmonotone two-sided line search ----------------
        alpha_p = alpha_m = 1.0
        accepted = False
        for _ in range(max_backtracks):
            x_new = x + alpha_p * d
            F_new = Feval(x_new)
            f_new = np.linalg.norm(F_new) ** 2
            if f_new <= fbar + eta - gamma * alpha_p ** 2 * fk:
                accepted = True
                break
            alpha_tp = alpha_p ** 2 * fk / (f_new + (2.0 * alpha_p - 1.0) * fk)

            x_new = x - alpha_m * d
            F_new = Feval(x_new)
            f_new = np.linalg.norm(F_new) ** 2
            if f_new <= fbar + eta - gamma * alpha_m ** 2 * fk:
                accepted = True
                break
            alpha_tm = alpha_m ** 2 * fk / (f_new + (2.0 * alpha_m - 1.0) * fk)

            alpha_p = float(np.clip(alpha_tp, tau_min * alpha_p, tau_max * alpha_p))
            alpha_m = float(np.clip(alpha_tm, tau_min * alpha_m, tau_max * alpha_m))
        if not accepted:
            status = "line_search_failed"
            break

        # ---------------- spectral coefficient and update ----------------
        s = x_new - x
        y = F_new - Fk
        sy = s @ y
        sigma = (s @ s) / sy if sy != 0.0 else np.inf   # undefined quotient: out of range, so the safeguard below replaces it
        x, Fk, fk = x_new, F_new, f_new
        history.append(fk)
        if len(history) > M:
            history.pop(0)
        k += 1

    # convergence is decided from the final stored residual, whichever way the loop ended
    if status == "maxit" and np.sqrt(fk) <= tol:
        status = "converged"

    return dict(x=x, itr=k, nf=nf, normF=float(np.sqrt(fk)), status=status)

## 4. DPRP+

```
DPRP+ : diagonal Polak-Ribiere-Polyak-like derivative-free method for F(x) = 0,
H. Mohammad, A. M. Awwal, Numerical Algorithms 91 (2022) 1441-1460, Algorithm 1.
Equation numbers below are those of that paper.

Merit function (2):   f(x) = 0.5 ||F(x)||^2
Iteration (4):        x_{k+1} = x_k + lambda_k d_k
Direction (7):        d_0 = -F_0 ;   d_k = -Q_k F_k + beta_k d_{k-1}   (k >= 1)
Diagonal matrix (8):  Q_k = diag(1/q_k^1, ..., 1/q_k^n)
Entries (9):          q_k^i = max{ min{ y_{k-1}^i / s_{k-1}^i , u }, l }   if s_{k-1}^i != 0,   else 1
                      with y_{k-1} = F_k - F_{k-1},  s_{k-1} = x_k - x_{k-1},   0 < l < 1 <= u
Parameter (10):       beta_k = max{0, <F_k, y_{k-1}>} / max{ ||F_{k-1}||^2 , ||d_{k-1}|| }
                      (the denominator is max of the SQUARED norm of F_{k-1} and the norm (not squared)
                       of d_{k-1}, exactly as printed in the paper)
Line search (12):     lambda_k = theta rho^i, i the least integer >= 0 with
                      f(x_k + lambda d_k) <= (1 + tau_k) f(x_k) - sigma lambda^2 ||d_k||^2

Parameters
  from the paper (Section 4):  theta = 1, rho = 0.2, tau_k = 1 / e^{(k+1)^(k+1)}
  tau_k = e^{-(k+1)^(k+1)} (the PDF prints 1/e^{k+1^{k+1}} with ambiguous grouping; this reading was confirmed by
  the user).  Default tau_rule="power" (summable; underflows to exactly 0 for k >= 4, so late iterations are monotone in floating point).
  tau_rule="literal" parses the printed exponent literally, k + 1^(k+1) = k+1, i.e. e^{-(k+1)}.  Both are summable;
  on a1a.t, a9a.t, colon-cancer the two readings change individual runs slightly, not the conclusions.
  NOT given in the paper (my choices):  sigma = 1e-4, l = 1e-2, u = 1e2
  (the paper only requires 0 < l < 1 <= u and sigma in (0,1))

Signs: eq. (4) x_{k+1} = x_k + lambda_k d_k, eq. (7) d_k = -Q_k F_k + beta_k d_{k-1}, the line search (12) f(x_k + theta rho^i d_k)
and Step 4 of Algorithm 1 (x_{k+1} = x_k + lambda_k d_k) all use the plus convention in the copy of the PDF used here; no
sign inconsistency was found, and the code follows these equations as printed.

Counting: nf = number of evaluations of F (including the one at x_0).  Stopping: ||F(x_k)|| <= tol.
```

In [ ]:
import math

import numpy as np


def dprpplus(F, x0, theta=1.0, rho=0.2, sigma=1e-4, ell=1e-2, u=1e2,
             tol=1e-6, maxit=2000, max_backtracks=200, tau_rule="power", trace=None):
    assert tau_rule in ("power", "literal")
    nf = 0

    def Feval(x):
        nonlocal nf
        nf += 1
        return F(x)

    x = np.array(x0, dtype=float)
    Fk = Feval(x)
    status = "maxit"
    k = 0
    d = None
    x_prev = F_prev = None

    while k < maxit:
        normF = np.linalg.norm(Fk)
        if trace is not None:
            trace.append(float(normF))   # residual of the current iterate (diagnostic only)
        if normF <= tol:
            break

        # ---- direction, eqs (7)-(10)
        if k == 0:
            d = -Fk
        else:
            s = x - x_prev                                  # s_{k-1}
            y = Fk - F_prev                                 # y_{k-1}
            q = np.ones_like(s)                             # (9): q = 1 where s^i = 0
            nz = s != 0.0
            q[nz] = np.maximum(np.minimum(y[nz] / s[nz], u), ell)
            beta = max(0.0, float(Fk @ y)) / max(float(F_prev @ F_prev), float(np.linalg.norm(d)))   # (10)
            d = -Fk / q + beta * d                          # (7), (8): Q_k F_k = F_k / q

        # ---- line search (12): f = 0.5 ||F||^2
        if tau_rule == "power":                              # 1 / e^{(k+1)^(k+1)}  (underflows to 0 for k >= 4)
            tau_k = 0.0 if (k + 1) ** (k + 1) > 700 else math.exp(-float((k + 1) ** (k + 1)))
        else:                                               # literal parse of the printed exponent "k + 1^(k+1)" = k + 1:  1 / e^{k+1}
            tau_k = math.exp(-(k + 1.0))
        fk = 0.5 * (Fk @ Fk)
        dd = float(d @ d)
        lam = theta
        accepted = False
        for _ in range(max_backtracks):
            x_new = x + lam * d
            F_new = Feval(x_new)
            if 0.5 * (F_new @ F_new) <= (1.0 + tau_k) * fk - sigma * lam ** 2 * dd:
                accepted = True
                break
            lam *= rho
        if not accepted:
            status = "line_search_failed"
            break

        x_prev, F_prev = x, Fk
        x, Fk = x_new, F_new
        k += 1

    normF = float(np.linalg.norm(Fk))
    if status == "maxit" and normF <= tol:
        status = "converged"
    return dict(x=x, itr=k, nf=nf, normF=normF, status=status)

## 5. GN-BFGS

```
GN-BFGS : Gauss-Newton-based BFGS method for symmetric nonlinear equations g(x) = 0,
D.-H. Li, M. Fukushima, SIAM J. Numer. Anal. 37 (1999) 152-172, Algorithm 1.
(Applicable here because F = grad f has a symmetric Jacobian.)  Equation numbers are the paper's.

Step 0  x_0, B_0 = I, {eps_k} summable, r, rho in (0,1), sigma1, sigma2 > 0, lambda_{-1} > 0.
Step 1  stop if g_k = 0, else solve   B_k p + (1/lambda_{k-1}) ( g(x_k + lambda_{k-1} g_k) - g_k ) = 0       (12)
Step 2  if ||g(x_k + p_k)|| <= rho ||g_k||                                                                  (13)
            then lambda_k = 1
Step 3  else lambda_k = r^i, i the least integer >= 0 with
            ||g(x_k + lambda p_k)||^2 - ||g_k||^2 <= - sigma1 ||lambda g_k||^2 - sigma2 ||lambda p_k||^2 + eps_k ||g_k||^2   (11)
Step 4  x_{k+1} = x_k + lambda_k p_k
Step 5  s_k = x_{k+1} - x_k,  delta_k = g_{k+1} - g_k,  y_k = g(x_k + delta_k) - g(x_k)
        if y_k's_k <= 0 : B_{k+1} = B_k, else BFGS update (14):
            B_{k+1} = B_k - B_k s s' B_k / (s' B_k s) + y y' / (y's)

Parameters, as in the paper's experiments (Section 4, p. 169):  r = 0.1, rho = sqrt(0.9) = 0.94868..., sigma1 = sigma2 = 1e-5,
lambda_{-1} = 0.01, eps_k = k^-2, B_0 = I.  The paper's eps_k = k^-2 is undefined at k = 0, so
eps_k = (k+1)^-2 is used (my choice for that index).

B_k is stored as a dense n x n matrix (n = 122 or 123 for a1a.t..a9a.t, n = 2000 for colon-cancer) and (12) is solved
with a dense Cholesky factorisation (B_k is symmetric positive definite by Step 5).  Cost per iteration is O(n^3) for
the factorisation, which the counter NF does not include; the CPU time does.

Counting: nf = every evaluation of g, i.e. g(x_0), g(x_k + lambda_{k-1} g_k) in (12), the trial g(x_k + p_k) in (13),
each line-search trial of (11), and g(x_k + delta_k) in Step 5.  A trial point that is accepted is reused as g_{k+1}.
Stopping: ||g(x_k)|| <= tol.
```

In [ ]:
import numpy as np
import scipy.linalg as sla


def gnbfgs(F, x0, r=0.1, rho=0.9486832980505138, sigma1=1e-5, sigma2=1e-5, lam_prev=0.01,
           tol=1e-6, maxit=2000, max_backtracks=60, trace=None):
    nf = 0

    def g(x):
        nonlocal nf
        nf += 1
        return F(x)

    x = np.array(x0, dtype=float)
    n = x.size
    gk = g(x)
    B = np.eye(n)
    status = "maxit"
    k = 0
    n_skipped = 0

    while k < maxit:
        ng = np.linalg.norm(gk)
        if trace is not None:
            trace.append(float(ng))   # residual of the current iterate (diagnostic only)
        if ng <= tol:
            break

        # Step 1, eq (12)
        rhs = -(g(x + lam_prev * gk) - gk) / lam_prev
        try:
            p = sla.cho_solve(sla.cho_factor(B), rhs)
        except np.linalg.LinAlgError:
            status = "singular_B"
            break

        # Step 2, eq (13)
        x_new = x + p
        g_new = g(x_new)
        if np.linalg.norm(g_new) <= rho * ng:
            lam = 1.0
        else:
            # Step 3, eq (11)
            eps_k = 1.0 / (k + 1) ** 2
            lam, accepted = 1.0, False
            for i in range(max_backtracks):
                if i > 0:                                  # i = 0 (lambda = 1) reuses the trial point of (13)
                    lam *= r
                    x_new = x + lam * p
                    g_new = g(x_new)
                lhs = g_new @ g_new - gk @ gk
                rhs11 = -sigma1 * lam ** 2 * (gk @ gk) - sigma2 * lam ** 2 * (p @ p) + eps_k * (gk @ gk)
                if lhs <= rhs11:
                    accepted = True
                    break
            if not accepted:
                status = "line_search_failed"
                break

        # Step 5
        s = x_new - x
        delta = g_new - gk
        y = g(x + delta) - gk
        ys = float(y @ s)
        if ys > 0.0:
            Bs = B @ s
            B = B - np.outer(Bs, Bs) / float(s @ Bs) + np.outer(y, y) / ys     # (14)
        else:
            n_skipped += 1

        lam_prev = lam
        x, gk = x_new, g_new
        k += 1

    normF = float(np.linalg.norm(gk))
    if status == "maxit" and normF <= tol:
        status = "converged"
    return dict(x=x, itr=k, nf=nf, normF=normF, status=status, n_skipped=n_skipped)

## 6. Sanity checks, validation of DF-SANE and regression tests

1. `check_gradient()` — asserts that `F(x)` equals the numerical gradient of `f`, and that every method reports `converged`
   with `||F|| <= tol` and ends within `2*tol/reg` of the others (strong convexity bound).
2. `check_dfsane()` — compares `dfsane.py` (variant "scipy") with `scipy.optimize.root(method='df-sane')`.
3. `run_edge_case_tests()` — regression tests: status at the iteration limit, DF-SANE with `s'y = 0`, and every method on a small SPD linear system.

In [ ]:
import numpy as np
import scipy.sparse as sp


def check_gradient():
    """Gradient check and 'same solution' check."""
    rng = np.random.default_rng(1)
    N, n = 500, 20
    A = sp.random(N, n, density=0.3, random_state=0, format="csr")
    b = np.where(rng.random(N) < 0.5, 1.0, -1.0)
    P = LogisticProblem(A, b)

    # (1) central finite differences of f versus F.  Expected error: about eps*|f|/e + e^2 |f'''|, ~1e-10.
    x = rng.normal(size=n)
    e = 1e-6
    num = np.array([(P.f(x + e * np.eye(n)[i]) - P.f(x - e * np.eye(n)[i])) / (2 * e) for i in range(n)])
    err = np.abs(P.F(x) - num).max()
    print("max |F(x) - numerical gradient| :", err)
    assert err < 1e-8, f"gradient check failed: {err}"

    # (2) all methods reach the same solution
    tol = 1e-9
    x0 = rng.normal(size=n)
    runs = {
        "RHP": rhp(P.F, x0, tol=tol),
        "DF-SANE": dfsane(P.F, x0, tol=tol),
        "DPRP+": dprpplus(P.F, x0, tol=tol),
        "DPRP+(.5,1)": dprpplus(P.F, x0, ell=0.5, u=1.0, tol=tol),
        "GN-BFGS": gnbfgs(P.F, x0, tol=tol),
    }
    ref = runs["RHP"]["x"]
    for name, r in runs.items():
        res = np.linalg.norm(P.F(r["x"]))
        diff = np.linalg.norm(r["x"] - ref)
        print(f"{name:11s} status={r['status']:9s} ||x - x_RHP|| = {diff:.2e}   ||F(x)|| = {res:.2e}")
        assert r["status"] == "converged", (name, r["status"])
        assert res <= tol, (name, res)
        assert diff <= 2 * tol / P.reg, (name, diff)
    print("gradient and same-solution checks passed")

In [ ]:
import numpy as np


def run_edge_case_tests():
    """Regression tests for the issues found in the independent review."""
    # 1. status must say "converged" if the last allowed iteration reached the tolerance (RHP), and for a start
    #    that already is a root.
    r1 = rhp(lambda x: x, np.array([1.0]), maxit=1)
    r2 = rhp(lambda x: x, np.array([0.0]), maxit=0)
    assert r1["status"] == r2["status"] == "converged", (r1, r2)
    print("1. status at iteration limit               : converged (RHP, initial root)")

    # 2. DF-SANE with s'y = 0 (undefined spectral quotient).  F(x) = 1 + max(x-1,0)^2 - max(-x-1,0)^2, x0 = 0:
    #    the first accepted step lands on x = -1 with F unchanged (= 1), so s'y = 0.  The undefined quotient must be
    #    passed through the safeguard of the chosen variant (paper: sigma = 1, next trial x = -2, the root),
    #    not turned into the maximal step 1e10 (23 evaluations of F instead of 3).
    trials = []
    def F_flat(x):
        t = float(x[0]); trials.append(t)
        return np.array([1.0 + max(t - 1, 0) ** 2 - max(-t - 1, 0) ** 2])
    r = dfsane(F_flat, [0.0], tol=1e-6)
    assert r["status"] == "converged" and r["nf"] == 3 and abs(trials[2] + 2.0) < 1e-12, (r, trials[:4])
    print("2. DF-SANE zero curvature -> safeguard    :", r["nf"], "evaluations of F, next trial x =", trials[2])

    # 3. Every method solves a small, well-conditioned symmetric linear system F(x) = H x - c = 0 (H symmetric positive definite).
    rng = np.random.default_rng(0)
    M = rng.normal(size=(30, 30)); H = M @ M.T / 30 + np.eye(30); c = rng.normal(size=30)
    Flin = lambda x: H @ x - c
    xs = np.linalg.solve(H, c)
    for name, run in {"RHP": lambda: rhp(Flin, np.zeros(30), tol=1e-6),
                      "DF-SANE": lambda: dfsane(Flin, np.zeros(30), tol=1e-6),
                      "DPRP+": lambda: dprpplus(Flin, np.zeros(30), tol=1e-6),
                      "GN-BFGS": lambda: gnbfgs(Flin, np.zeros(30), tol=1e-6)}.items():
        r = run()
        assert r["status"] == "converged" and np.linalg.norm(r["x"] - xs) <= 1e-6 * 2 * np.linalg.norm(H, 2), (name, r["status"], r["itr"])
        print(f"3. linear system, {name:10s} converged    :", r["itr"], "iterations")
    print("all edge-case tests passed")

In [ ]:
import numpy as np
from scipy.optimize import root


def check_dfsane():
    """Compare dfsane.py with scipy.optimize.root(method='df-sane') on every data set."""
    TOL = 1e-6
    worst_dx, mismatches, total = 0.0, 0, 0
    print(f"{'dataset':13s}{'own itr/nf':>16s}{'scipy itr/nf':>16s}   max |x_own - x_scipy|")
    for name in list(DATASETS):
        A, b = load_libsvm(DATA_DIR, name)
        P = LogisticProblem(A, b)
        rng = np.random.default_rng(0)
        starts = [4.0 * (rng.random(P.n) - 0.5) for _ in range(5)]
        own_i, own_f, sp_i, sp_f, dx = [], [], [], [], 0.0
        for x0 in starts:
            r = dfsane(P.F, x0, tol=TOL, variant="scipy")
            s = root(P.F, x0, method="df-sane",
                     options=dict(ftol=0.0, fatol=TOL, maxfev=10**7, line_search="cruz"))
            total += 1
            same = (r["status"] == "converged") and s.success and r["itr"] == s.nit and r["nf"] == s.nfev
            mismatches += not same
            d = np.linalg.norm(r["x"] - s.x)
            dx = max(dx, d)
            own_i.append(r["itr"]); own_f.append(r["nf"]); sp_i.append(s.nit); sp_f.append(s.nfev)
            assert r["status"] == "converged" and s.success, (name, r["status"], s.message)
            assert d <= 2 * TOL / P.reg, (name, d)          # both are within tol/reg of the unique root
        worst_dx = max(worst_dx, dx)
        print(f"{name:13s}{np.mean(own_i):9.1f}/{np.mean(own_f):<6.0f}{np.mean(sp_i):9.1f}/{np.mean(sp_f):<6.0f}   {dx:.2e}")
    print(f"\nruns with different (itr, nf) than SciPy: {mismatches} of {total};  worst solution difference {worst_dx:.2e}")
    assert mismatches == 0, "own DF-SANE does not reproduce SciPy's iteration/evaluation counts"
    print("dfsane.py reproduces scipy.optimize.root(method='df-sane') exactly")

In [ ]:
check_gradient()
print()
check_dfsane()
print()
run_edge_case_tests()

max |F(x) - numerical gradient| : 1.8618908801878997e-10
RHP         status=converged ||x - x_RHP|| = 0.00e+00   ||F(x)|| = 8.59e-10
DF-SANE     status=converged ||x - x_RHP|| = 6.11e-09   ||F(x)|| = 4.01e-10
DPRP+       status=converged ||x - x_RHP|| = 8.12e-09   ||F(x)|| = 6.72e-10
DPRP+(.5,1) status=converged ||x - x_RHP|| = 1.07e-08   ||F(x)|| = 9.71e-10
GN-BFGS     status=converged ||x - x_RHP|| = 7.31e-09   ||F(x)|| = 6.35e-10
gradient and same-solution checks passed

dataset            own itr/nf    scipy itr/nf   max |x_own - x_scipy|
a1a.t            185.8/453       185.8/453      0.00e+00
a2a.t            188.4/459       188.4/459      0.00e+00
a3a.t            188.0/459       188.0/459      0.00e+00
a4a.t            190.8/464       190.8/464      0.00e+00
a5a.t            184.0/448       184.0/448      0.00e+00
a6a.t            172.2/416       172.2/416      0.00e+00
a7a.t            202.8/497       202.8/497      0.00e+00
a8a.t             69.6/142        69.6/142      0.00

## 7. The experiment

5 random starts per data set, identical stopping rule for every method.

```
Logistic-regression experiment (Section 5.2 protocol of the inertial paper):
restarted hybridplus (RHP), DF-SANE, DPRP+ (two (l,u) settings) and GN-BFGS.

  * regularisation weight 0.1
  * 5 runs per data set; starting points 4*(rand(n)-0.5) (uniform in [-2, 2]^n).
    The numpy generator is re-seeded with 0 for EVERY data set, so data sets with the same number
    of variables n (a1a.t..a7a.t: n=123; a8a.t, a9a.t: n=122) use the same five starting vectors.
    Within a data set all methods use the same five vectors (paired comparison).  numpy's stream is
    not MATLAB's rand stream, so starting points differ from those of the paper.
  * stop when ||F(x_k)||_2 <= 1e-6, or after 2000 iterations
  * per run we record: iterations, NF, process CPU time (and wall time), final ||F||, status.
    Summary = average over the 5 runs.

Columns of the printed table  (averages):   Itr / NF / CPU-seconds  solved
  NF   number of evaluations of F.
  CPU  process CPU time (time.process_time); data loading excluded.  The elapsed wall-clock time
       (time.perf_counter) is also stored per run.  The methods run one after another in a fixed
       order on one machine.
```

In [ ]:
import hashlib
import inspect
import json
import os
import platform
import sys
import time

import numpy as np
import scipy
import sklearn


TOL = 1e-6
MAXIT = 2000
N_RUNS = 5


def system_info():
    """Hardware / software configuration of the machine the experiment runs on (Colab or local); nothing is hard-coded."""
    import re
    import shutil
    info = dict(python=sys.version.split()[0], numpy=np.__version__, scipy=scipy.__version__,
                sklearn=sklearn.__version__, platform=platform.platform(), machine=platform.machine())
    info["colab"] = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
    try:
        with open("/proc/cpuinfo") as fh:
            txt = fh.read()
        m = re.search(r"model name\s*:\s*(.+)", txt)
        info["cpu_model"] = re.sub(r"\s+", " ", m.group(1)).strip() if m else platform.processor()
    except Exception:
        info["cpu_model"] = platform.processor() or "unknown"
    try:
        info["logical_cpus"] = len(os.sched_getaffinity(0))
    except Exception:
        info["logical_cpus"] = os.cpu_count()
    try:
        with open("/proc/meminfo") as fh:
            kb = int(re.search(r"MemTotal:\s+(\d+)", fh.read()).group(1))
        info["ram_gb"] = round(kb / 1024 ** 2, 1)
    except Exception:
        info["ram_gb"] = None
    try:
        with open("/etc/os-release") as fh:
            info["os"] = re.search(r'PRETTY_NAME="?([^"\n]+)', fh.read()).group(1)
    except Exception:
        info["os"] = platform.system() + " " + platform.release()
    info["accelerator"] = "none"
    if shutil.which("nvidia-smi"):
        try:
            import subprocess
            out = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                                 capture_output=True, text=True, timeout=10).stdout.strip()
            info["accelerator"] = out.splitlines()[0] if out else "none"
        except Exception:
            pass
    try:
        from threadpoolctl import threadpool_info
        info["blas"] = [dict(api=p.get("internal_api"), threads=p.get("num_threads")) for p in threadpool_info()
                        if p.get("user_api") == "blas"]
    except Exception:
        info["blas"] = []
    return info


def effective_params(fn, **override):
    """Keyword values a solver actually runs with (signature defaults, then overrides); used to document the setup."""
    skip = {"F", "x0", "tol", "maxit", "trace", "max_backtracks", "variant", "tau_rule"}
    p = {k: v.default for k, v in inspect.signature(fn).parameters.items()
         if v.default is not inspect.Parameter.empty and k not in skip}
    p.update({k: v for k, v in override.items() if k in p})
    return p


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def find_file(data_dir, name):
    for suffix in ("", ".txt", ".bz2"):
        p = os.path.join(data_dir, name + suffix)
        if os.path.isfile(p):
            return p


def summarise(records):
    cpu = np.array([r["cpu"] for r in records])
    return dict(
        itr=float(np.mean([r["itr"] for r in records])),
        nf=float(np.mean([r["nf"] for r in records])),
        cpu=float(cpu.mean()), cpu_std=float(cpu.std(ddof=1)),
        wall=float(np.mean([r["wall"] for r in records])),
        normF=float(np.mean([r["normF"] for r in records])),
        solved=sum(r["normF"] <= TOL for r in records),
        n_status=sorted({r["status"] for r in records}),
    )


def run_all(data_dir, dataset_names=None, out="results.json", starts_out="starts.npz"):
    dataset_names = dataset_names or list(DATASETS)
    # (function, keyword overrides): the settings are written to results.json and used by the report generator
    spec = {
        "RHP":         (rhp, {}),                          # defaults of rhp(): t=0.8, rho=0.85, sigma=1e-4, lmax=10, ...
        "DF-SANE":     (dfsane, {}),
        "DPRP+":       (dprpplus, {}),                     # (ell, u) = (0.01, 100)
        "DPRP+(.5,1)": (dprpplus, dict(ell=0.5, u=1.0)),
        "GN-BFGS":     (gnbfgs, {}),
    }
    methods = {m: (lambda P, x0, tr=None, fn=fn, kw=kw: fn(P.F, x0, tol=TOL, maxit=MAXIT, trace=tr, **kw))
               for m, (fn, kw) in spec.items()}
    settings = {m: effective_params(fn, **kw) for m, (fn, kw) in spec.items()}

    env = system_info()
    env.update(tol=TOL, maxit=MAXIT, n_runs=N_RUNS, reg=REG,
               timer="CPU time: time.process_time; wall: time.perf_counter")
    datasets, starts_store, traces = [], {}, {}

    print(f"{'dataset':13s}| " + " | ".join(f"{m:>26.26s}" for m in methods) + "    (Itr/NF/CPU-s solved)")
    for name in dataset_names:
        path = find_file(data_dir, name)
        A, b = load_libsvm(data_dir, name)
        P = LogisticProblem(A, b)
        rng = np.random.default_rng(0)
        starts = [4.0 * (rng.random(P.n) - 0.5) for _ in range(N_RUNS)]
        starts_store[name] = np.array(starts)

        per_run, summary = {}, {}
        for mname, solve in methods.items():
            recs = []
            for j, x0 in enumerate(starts):
                tr = [] if j == 0 else None          # residual history of the first start only (diagnostic, O(n) per iteration)
                t0, c0 = time.perf_counter(), time.process_time()
                r = solve(P, x0, tr)
                wall, cpu = time.perf_counter() - t0, time.process_time() - c0
                if tr is not None:
                    traces.setdefault(name, {})[mname] = dict(trace=tr, itr=r["itr"], status=r["status"], normF=r["normF"])
                recs.append(dict(itr=r["itr"], nf=r["nf"], wall=wall, cpu=cpu,
                                 normF=r["normF"], status=r["status"],
                                 restarts=r.get("restarts"), n_skipped=r.get("n_skipped")))
            per_run[mname] = recs
            summary[mname] = summarise(recs)

        cells = [f"{s['itr']:.1f}/{s['nf']:.0f}/{s['cpu']:.3f} {s['solved']}/{N_RUNS}"
                 for s in summary.values()]
        print(f"{name:13s}| " + " | ".join(f"{c:>26s}" for c in cells), flush=True)
        datasets.append(dict(dataset=name, N=P.N, n=P.n, file=os.path.basename(path),
                             sha256=sha256(path), summary=summary, per_run=per_run))

    with open(out, "w") as fh:
        json.dump(dict(environment=env, settings=settings, datasets=datasets), fh, indent=1)
    with open(os.path.splitext(out)[0] + "_traces.json", "w") as fh:
        json.dump(traces, fh)
    np.savez(starts_out, **starts_store)

In [ ]:
# Runs the full experiment (roughly 10-15 minutes) and writes results.json and starts.npz next to the notebook.
# For a quick test use e.g.  run_all(DATA_DIR, ["a8a.t", "colon-cancer"])
run_all(DATA_DIR)

dataset      |                        RHP |                    DF-SANE |                      DPRP+ |                DPRP+(.5,1) |                    GN-BFGS    (Itr/NF/CPU-s solved)
a1a.t        |          24.4/66/0.207 5/5 |         73.2/154/0.464 5/5 |        221.8/856/2.084 5/5 |          70.6/78/0.165 5/5 |         71.6/216/0.456 5/5
a2a.t        |          25.4/70/0.135 5/5 |         67.4/136/0.252 5/5 |        228.2/887/1.876 5/5 |          69.8/76/0.152 5/5 |         71.6/216/0.452 5/5
a3a.t        |          24.6/70/0.125 5/5 |         73.4/154/0.278 5/5 |        235.2/910/1.861 5/5 |          68.8/74/0.131 5/5 |         71.4/215/0.412 5/5
a4a.t        |          25.2/66/0.123 5/5 |         67.4/135/0.267 5/5 |        213.8/824/1.690 5/5 |          71.0/78/0.135 5/5 |         71.6/216/0.405 5/5
a5a.t        |          25.8/80/0.123 5/5 |         66.2/134/0.207 5/5 |        219.4/850/1.643 5/5 |          70.0/77/0.129 5/5 |         71.0/214/0.357 5/5
a6a.t        |          25.

## 8. Results summary (read back from `results.json`)

In [ ]:
import json

with open("results.json") as fh:
    res = json.load(fh)

print(f"{'dataset':13s} {'method':11s} {'Itr':>7s} {'NF':>7s} {'CPU s':>8s} {'+-std':>7s} {'solved':>7s}")
for d in res["datasets"]:
    for m, s in d["summary"].items():
        print(f"{d['dataset']:13s} {m:11s} {s['itr']:7.1f} {s['nf']:7.0f} "
              f"{s['cpu']:8.3f} {s['cpu_std']:7.3f} {s['solved']:>5d}/5")
    print()
print("environment:", res["environment"])

dataset       method          Itr      NF    CPU s   +-std  solved
a1a.t         RHP            24.4      66    0.207   0.034     5/5
a1a.t         DF-SANE        73.2     154    0.464   0.471     5/5
a1a.t         DPRP+         221.8     856    2.084   0.542     5/5
a1a.t         DPRP+(.5,1)    70.6      78    0.165   0.009     5/5
a1a.t         GN-BFGS        71.6     216    0.456   0.006     5/5

a2a.t         RHP            25.4      70    0.135   0.029     5/5
a2a.t         DF-SANE        67.4     136    0.252   0.336     5/5
a2a.t         DPRP+         228.2     887    1.876   0.425     5/5
a2a.t         DPRP+(.5,1)    69.8      76    0.152   0.014     5/5
a2a.t         GN-BFGS        71.6     216    0.452   0.025     5/5

a3a.t         RHP            24.6      70    0.125   0.041     5/5
a3a.t         DF-SANE        73.4     154    0.278   0.283     5/5
a3a.t         DPRP+         235.2     910    1.861   0.511     5/5
a3a.t         DPRP+(.5,1)    68.8      74    0.131   0.011  

## 9. Report generation (`temp/report.tex`)

```
Report generator: results.json (+ traces) -> temp/report.tex, figures, supporting files, temp.zip.

The report text is the TEMPLATE (report_template.tex); this module fills in everything that depends on the experiment:
solver settings, data-set sizes, system configuration, tables, figures, and every number or data-dependent statement
of the results and the discussion.  Static prose is never rewritten.  A statement whose truth depends on the data
(e.g. "X is the cheapest method") is produced from the data, or omitted when it does not hold.

    make_report(results_path, traces_path, outdir="temp", template="report_template.tex")
```

The template of the next cell is the body of `temp/report.tex` of the Overleaf project (heading, text, tables, figures, discussion), with placeholders; `report.tex` is meant for `\input{temp/report}` in the manuscript, and `report_standalone.tex` (the preamble and bibliography of the original file) compiles it on its own. Set `NAME_STYLE = "manuscript"` in the generator cell to print the method names as `\texttt{dfsane}`, `\texttt{dprpplus}`, `\texttt{gnbfgs}` instead of DF-SANE, DPRP$^+$, GN-BFGS. Everything that depends on the results is filled in from `results.json`: parameter tables, data-set sizes, system configuration, the four tables, both figures, and every number and data-dependent statement of the results and the discussion. Nothing else in the report is rewritten.

In [ ]:
import json
import math
import os
import re
import shutil

import matplotlib
import numpy as np

matplotlib.use("Agg")
import matplotlib.pyplot as plt

METHODS = ["RHP", "DF-SANE", "DPRP+", "DPRP+(.5,1)", "GN-BFGS"]
RHP, DF, DA, DB, GN = METHODS
# Names of the methods.  "report" = names of the report template (RHP, DF-SANE, DPRP+, GN-BFGS);
# "manuscript" = \texttt names of the MATLAB subsection (applied as a final text substitution, see apply_names).
NAME_STYLE = "report"
PROSE = {RHP: "RHP", DF: "DF-SANE", DA: r"DPRP$^{+}_{\mathrm A}$", DB: r"DPRP$^{+}_{\mathrm B}$", GN: "GN-BFGS"}
BASE = {RHP: PROSE[RHP], DF: PROSE[DF], DA: r"DPRP$^{+}$", DB: r"DPRP$^{+}$", GN: PROSE[GN]}
HEAD = dict(PROSE)
PLOT_NAMES = {"report": {RHP: "RHP", DF: "DF-SANE", DA: "DPRP$^{+}_{A}$", DB: "DPRP$^{+}_{B}$", GN: "GN-BFGS"},
              "manuscript": {RHP: "restarted hybridplus", DF: "dfsane", DA: "dprpplus$_A$", DB: "dprpplus$_B$", GN: "gnbfgs"}}
PLOT = PLOT_NAMES["report"]


def apply_names(text, style):
    if style != "manuscript":
        return text
    for a, b in [(r"DPRP$^{+}_{\mathrm A}$", r"\texttt{dprpplus}$_{\mathrm A}$"), (r"DPRP$^{+}_{\mathrm B}$", r"\texttt{dprpplus}$_{\mathrm B}$"),
                 (r"DPRP$^{+}$", r"\texttt{dprpplus}"), ("DF-SANE", r"\texttt{dfsane}"), ("GN-BFGS", r"\texttt{gnbfgs}")]:
        text = text.replace(a, b)
    return text


COL = dict(zip(METHODS, ["#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6"]))
LS = dict(zip(METHODS, ["-", "--", "-.", ":", "-"]))
# parameter values of the nonlinear-equation experiments of the manuscript (Section "Solvers and parameters")
MANUSCRIPT_RHP = dict(t=0.95, rho=0.5, sigma=1e-2, lmin=1e-10, lmax=1e10, nu=1e-8)

WORDS = ["zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine", "ten", "eleven", "twelve",
         "thirteen", "fourteen", "fifteen", "sixteen", "seventeen", "eighteen", "nineteen", "twenty"]


def word(k):
    return WORDS[k] if 0 <= k < len(WORDS) else str(k)


def cap(s):
    return s[0].upper() + s[1:] if s and s[0].islower() else s


def num(x, nd=None):
    """Plain number; thin-space thousands separator for integers >= 10000."""
    if nd is None:
        s = f"{int(round(x)):,}".replace(",", r"\,") if abs(x) >= 10000 else f"{int(round(x))}"
        return s
    return f"{x:.{nd}f}"


def numsep(x):
    """Integer with a thin-space thousands separator from 1000 upwards (data-set sizes)."""
    x = int(round(x))
    return f"{x:,}".replace(",", r"\,") if x >= 1000 else str(x)


def sig(x, k=3):
    """k significant figures, trailing zeros removed (2.10 -> 2.1)."""
    if x == 0:
        return "0"
    d = k - 1 - int(math.floor(math.log10(abs(x))))
    s = f"{x:.{max(d, 0)}f}"
    return s.rstrip("0").rstrip(".") if "." in s else s


def tex_num(x):
    """LaTeX math for a parameter value: 1e-4 -> 10^{-4}, 0.01 -> 0.01, 100.0 -> 100, sqrt(0.9) -> \\sqrt{0.9}."""
    if isinstance(x, (int, np.integer)):
        return str(int(x))
    if abs(x - math.sqrt(0.9)) < 1e-12:
        return r"\sqrt{0.9}"
    if x == 0:
        return "0"
    e = math.floor(math.log10(abs(x)) + 1e-12)
    m = x / 10 ** e
    if abs(x) < 0.01 or abs(x) >= 1e5:
        if abs(m - 1) < 1e-9:
            return f"10^{{{e}}}"
        return f"{sig(m, 3)}\\times10^{{{e}}}"
    s = f"{x:.10g}"
    return s


def ratio_fmt(x):
    """Ratio formatting: one decimal from 2 upwards (3.3, 12.7), two decimals below (1.16)."""
    return f"{x:.1f}" if x >= 2 else f"{x:.2f}"


def sci(x, k=1):
    """a.b x 10^e in LaTeX."""
    e = int(math.floor(math.log10(abs(x))))
    m = x / 10 ** e
    return f"{m:.{k}f}\\times10^{{{e}}}"


def join_and(items):
    items = [i for i in items if i]
    if len(items) <= 1:
        return "".join(items)
    return ", ".join(items[:-1]) + " and " + items[-1]


# ----------------------------------------------------------------------------------------------------- statistics
class Stats:
    def __init__(self, R, T):
        self.R, self.T = R, T
        self.env = R["environment"]
        self.tol = self.env["tol"]
        self.maxit = self.env["maxit"]
        self.nr = self.env["n_runs"]
        self.ds = R["datasets"]
        isa = lambda d: re.fullmatch(r"a\d+a\.t", d["dataset"]) is not None
        self.A = [d for d in self.ds if isa(d)]
        self.C = [d for d in self.ds if not isa(d)]
        self.M = METHODS
        self.set = R["settings"]

    # per-data-set means
    def mean(self, group, m, key):
        return float(np.mean([d["summary"][m][key] for d in group]))

    def runs(self, group, m, key):
        out = []
        for d in group:
            out += [r[key] for r in d["per_run"][m]]
        return np.array(out, dtype=float)

    def ok(self, group, m):
        out = []
        for d in group:
            out += [r["normF"] <= self.tol for r in d["per_run"][m]]
        return np.array(out)

    def cost(self, group, key):
        """[n_runs, n_methods] cost matrix, inf for unsuccessful runs."""
        cols = []
        for m in self.M:
            v = self.runs(group, m, key)
            v[~self.ok(group, m)] = np.inf
            cols.append(v)
        return np.array(cols).T

    def per_iter(self, group, m):
        return self.mean(group, m, "nf") / self.mean(group, m, "itr")


def group_label(group):
    names = [d["dataset"] for d in group]
    if not names:
        return ""
    if len(names) == 1:
        return names[0]
    if all(re.fullmatch(r"a\d+a\.t", n) for n in names):
        return f"{names[0]}--{names[-1]}"
    return join_and(names)


# ------------------------------------------------------------------------------------------------------- tables
def make_tables(S):
    M = METHODS
    note_parts = []
    for d in S.ds:
        bad = [(m, d["summary"][m]["solved"]) for m in M if d["summary"][m]["solved"] < S.nr]
        if bad:
            note_parts.append(d["dataset"] + ": " + ", ".join(f"{PROSE[m]} {s} of {S.nr}" + (" runs converged" if i == 0 else "")
                                                              for i, (m, s) in enumerate(bad)))
    if note_parts:
        note = (r"Bold: smallest value in the row. $^{\dagger}$: the method did not reach $\|F\|\le" + tex_num(S.tol)
                + r"$ within " + str(S.maxit) + " iterations in some runs (" + "; ".join(note_parts)
                + "); the entries then include the unsuccessful runs at the iteration limit.")
    else:
        note = "Bold: smallest value in the row."

    def fmt(key, v):
        return f"{v:.3f}" if key == "cpu" else (f"{v:.1f}" if key == "itr" else f"{v:.0f}")

    def row(label, vals, solved=None, key=None):
        b = min(vals)
        cells = []
        for j, v in enumerate(vals):
            s = fmt(key, v)
            if solved is not None and solved[j] < S.nr:
                s += r"$^{\dagger}$"
            cells.append(r"\textbf{" + s + "}" if fmt(key, v) == fmt(key, b) else s)
        return label + " & " + " & ".join(cells) + r" \\"

    def table(key, cap_, label):
        rows = []
        for d in S.ds:
            vals = [d["summary"][m][key] for m in M]
            sol = [d["summary"][m]["solved"] for m in M]
            rows.append(row(d["dataset"].replace("_", r"\_"), vals, sol, key))
            if S.A and d is S.A[-1] and S.C:
                avg = [S.mean(S.A, m, key) for m in M]
                rows += [r"\midrule", row(r"\textit{mean, " + group_label(S.A) + "}", avg, None, key), r"\midrule"]
        return (f"\\begin{{table}}[H]\n\\centering\n\\caption{{{cap_}}}\n\\label{{{label}}}\n\\small\n"
                f"\\setlength{{\\tabcolsep}}{{4.5pt}}\n\\begin{{tabular}}{{l{'r' * len(M)}}}\n\\toprule\n"
                f"Data set & " + " & ".join(HEAD[m] for m in M) + " \\\\\n\\midrule\n" + "\n".join(rows)
                + f"\n\\bottomrule\n\\end{{tabular}}\n\\parbox{{0.97\\linewidth}}{{\\footnotesize {note}}}\n"
                "\\end{table}\n")

    t_itr = table("itr", f"Mean number of iterations over {word(S.nr)} random starting points.", "tab:logreg-itr")
    t_nf = table("nf", f"Mean number of evaluations of $F$ over {word(S.nr)} random starting points.", "tab:logreg-nf")
    t_cpu = table("cpu", f"Mean CPU time in seconds over {word(S.nr)} random starting points (data loading excluded).",
                  "tab:logreg-time")
    # summary table
    lines = []
    groups = [(group_label(S.A) + f" ({len(S.A) * S.nr} runs)", S.A)] if S.A else []
    if S.C:
        groups.append((group_label(S.C) + f" ({len(S.C) * S.nr} runs)", S.C))
    for lab, g in groups:
        lines.append(r"\multicolumn{" + str(len(M) + 1) + r"}{l}{\textit{" + lab + "}} \\\\")
        for key, nm in (("itr", "Iterations"), ("nf", "$F$-evaluations"), ("cpu", "CPU time (s)")):
            v = [S.mean(g, m, key) for m in M]
            b = min(v)
            cells = [(r"\textbf{" + fmt(key, x) + "}" if fmt(key, x) == fmt(key, b) else fmt(key, x)) for x in v]
            lines.append(nm + " & " + " & ".join(cells) + r" \\")
        lines.append(r"\midrule")
    lines = lines[:-1]
    fails_txt = ""
    for d in S.ds:
        bad = [(m, d["summary"][m]["solved"]) for m in M if d["summary"][m]["solved"] < S.nr]
        if bad:
            parts = [f"{PROSE[m]} converged in {s}" if i == 0 else f"{PROSE[m]} in {s}" for i, (m, s) in enumerate(bad)]
            fails_txt += f" On {d['dataset']}, " + " and ".join(parts) + f" of {S.nr} runs."
    t_sum = (r"\begin{table}[H]" + "\n\\centering\n"
             r"\caption{Aggregated results (means of the per-data-set means). $F$-evaluations count every evaluation of $F$ "
             r"(for GN-BFGS every evaluation of the residual map, including those in the finite-difference probes "
             r"and in Step~5). Bold: smallest value in the row." + fails_txt + "}\n"
             "\\label{tab:logreg-summary}\n\\small\n\\setlength{\\tabcolsep}{4.5pt}\n"
             "\\begin{tabular}{l" + "r" * len(M) + "}\n\\toprule\n & " + " & ".join(HEAD[m] for m in M)
             + " \\\\\n\\midrule\n" + "\n".join(lines) + "\n\\bottomrule\n\\end{tabular}\n\\end{table}\n")
    return "\n".join([t_itr, t_nf, t_cpu, t_sum])


# ------------------------------------------------------------------------------------------------------- figures
def profile_ratios(S, key):
    C = S.cost(S.ds, key)
    best = C.min(axis=1)
    return C / best[:, None]


def make_figures(S, outdir):
    plt.rcParams.update({"font.family": "serif", "font.size": 9, "axes.spines.top": False, "axes.spines.right": False})
    # performance profiles
    fig, axs = plt.subplots(1, 2, figsize=(7.4, 3.2))
    cap_ = 64
    taus = np.logspace(0, np.log2(cap_) * np.log10(2), 400)
    for ax, key, ttl in zip(axs, ("nf", "cpu"), ("(a) evaluations of $F$", "(b) CPU time")):
        r = profile_ratios(S, key)
        for j, m in enumerate(METHODS):
            ax.step(taus, [np.mean(r[:, j] <= t) for t in taus], where="post", color=COL[m], ls=LS[m], lw=1.6, label=PLOT[m])
        ax.set_xscale("log", base=2)
        ax.set_xlabel(r"performance ratio $\tau$")
        ax.set_ylim(0, 1.02)
        ax.grid(alpha=.3)
        ax.set_title(ttl, fontsize=9)
    axs[0].set_ylabel(r"fraction of runs with ratio $\leq\tau$")
    h, l = axs[0].get_legend_handles_labels()
    fig.legend(h, l, loc="lower center", ncol=5, frameon=False, fontsize=7.5)
    fig.subplots_adjust(bottom=0.33, left=0.09, right=0.99, top=0.9, wspace=0.18)
    fig.savefig(os.path.join(outdir, "fig_profiles.pdf"))
    plt.close(fig)
    # residual histories, first starting point of every data set
    names = [d["dataset"] for d in S.ds]
    nc = 5 if len(names) > 5 else len(names)
    nrw = int(math.ceil(len(names) / nc))
    plt.rcParams.update({"font.size": 7})
    fig, axs = plt.subplots(nrw, nc, figsize=(7.4, 2.2 * nrw), squeeze=False)
    axs = axs.ravel()
    xl_a = 0
    for d in S.A:
        for m in METHODS:
            xl_a = max(xl_a, len(S.T[d["dataset"]][m]["trace"]))
    xl_a = int(math.ceil(xl_a / 20.0) * 20) if xl_a else 0
    S.xl_a = xl_a
    for k, (ax, nm) in enumerate(zip(axs, names)):
        for m in METHODS:
            tr = S.T[nm][m]["trace"]
            ax.plot(range(len(tr)), tr, color=COL[m], ls=LS[m], lw=1.0, label=PLOT[m])
        ax.axhline(S.tol, color="gray", lw=.6, ls=":")
        ax.set_yscale("log")
        is_a = re.fullmatch(r"a\d+a\.t", nm) is not None
        ax.set_xlim(0, xl_a if is_a else S.maxit)
        ax.set_ylim(S.tol / 2, None)
        ax.grid(alpha=.3)
        ax.set_title(nm, fontsize=8)
        if k >= len(names) - nc:
            ax.set_xlabel("iteration $k$")
        if k % nc == 0:
            ax.set_ylabel(r"$\|F(x_k)\|$")
    for ax in axs[len(names):]:
        ax.axis("off")
    h, l = axs[0].get_legend_handles_labels()
    fig.legend(h, l, loc="lower center", ncol=5, frameon=False, fontsize=7)
    fig.subplots_adjust(bottom=0.2 if nrw == 2 else 0.3, left=0.07, right=0.99, top=0.94, wspace=0.35, hspace=0.4)
    fig.savefig(os.path.join(outdir, "fig_residual.pdf"))
    plt.close(fig)


# ------------------------------------------------------------------------------------------------- text blocks
def system_sentence(env):
    colab = env.get("colab")
    where = "in Google Colab" if colab else "on a local machine"      # same wording as the PLATFORM placeholder of the template
    os_ = env.get("os") or env.get("platform", "")
    cpu = env.get("cpu_model", "unknown CPU")
    n = env.get("logical_cpus")
    ram = env.get("ram_gb")
    acc = env.get("accelerator", "none")
    blas = []
    for b in env.get("blas", []):
        if b not in blas:
            blas.append(b)
    hw = f"{cpu}" + (f", {n} logical CPU{'s' if n != 1 else ''}" if n else "") + (f", {ram:g}\\,GB of RAM" if ram else "")
    sent = (f"All methods were run sequentially in a single Python process {where} ({os_}; {hw}; "
            + ("no accelerator was used" if acc == "none" else f"accelerator: {acc}, not used by the solvers") + "), with "
            f"Python~{env['python']}, NumPy~{env['numpy']}, SciPy~{env['scipy']} and scikit-learn~{env['sklearn']}")
    if blas:
        sent += "; the BLAS library (" + ", ".join(f"{b['api']}, {b['threads']} thread{'s' if b['threads'] != 1 else ''}"
                                                   for b in blas) + ") was used with its default threading"
    return sent + ". CPU times are process CPU times and therefore depend on this configuration; the counts of iterations and of evaluations of $F$ do not."


def data_paragraph(S):
    n_all = [d["n"] for d in S.ds]
    parts = []
    nsets = len(S.ds)
    head = f"We use the {word(nsets)} LIBSVM benchmark data sets \\cite{{changlin2011}} listed in \\cref{{tab:logreg-data}}: "
    if S.A:
        ns = sorted({d["n"] for d in S.A})
        nset = "$n\\in\\{" + ",".join(map(str, ns)) + "\\}$" if len(ns) > 1 else f"$n={ns[0]}$"
        Ns = [d["N"] for d in S.A]
        parts.append(f"the test files {group_label(S.A)} of the \\emph{{Adult}} collection, which are sparse with {nset} variables "
                     f"and between ${numsep(min(Ns))}$ and ${numsep(max(Ns))}$ observations")
    for d in S.C:
        parts.append(f"{d['dataset']}, which has $m={d['N']}$ observations but $n={d['n']}$ variables")
    txt = head + (", and ".join(parts) if len(parts) == 2 else join_and(parts)) + ". "
    if S.A and S.C:
        d = S.C[0]
        txt += f"The two groups probe different regimes: $m\\gg n$ for the a-sets, and $n\\gg m$ for {d['dataset']}."
    return txt.strip()


def data_table(S):
    ds = S.ds
    half = int(math.ceil(len(ds) / 2))
    left, right = ds[:half], ds[half:]
    rows = []
    for i in range(half):
        a = left[i]
        cell = f"{a['dataset']} & {numsep(a['N'])} & {numsep(a['n'])}"
        if i < len(right):
            b = right[i]
            cell += f" & {b['dataset']} & {numsep(b['N'])} & {numsep(b['n'])}"
        else:
            cell += " & & & "
        rows.append(cell + r" \\")
    return "lrrlrr", r"Data set & $m$ & $n$ & Data set & $m$ & $n$", "\n".join(rows)


def success_paragraph(S):
    M = METHODS
    tot = len(S.ds) * S.nr * len(M)
    nok = sum(S.ok([d], m).sum() for d in S.ds for m in M)
    # failures grouped by data set
    bad_methods, clauses = set(), []
    for d in S.ds:
        items = []
        for m in M:
            recs = d["per_run"][m]
            fl = [r for r in recs if r["normF"] > S.tol]
            if fl:
                bad_methods.add(m)
                res = sorted(r["normF"] for r in fl)
                how = "stopped at the iteration limit" if all(r["status"] == "maxit" for r in fl) else "did not reach the tolerance"
                k = len(fl)
                cnt = f"in all {word(S.nr)} runs" if k == S.nr else (f"in one run" if k == 1 else f"in {word(k)} runs")
                rs = (f"final residual ${sci(res[0])}$" if k == 1 else
                      f"final residuals between ${sci(res[0])}$ and ${sci(res[-1])}$")
                items.append((m, f"{PROSE[m]} {how} {cnt} ({rs})"))
        if items:
            bases = []
            for m, _ in items:
                if BASE[m] not in bases:
                    bases.append(BASE[m])
            first = items[0][1]
            rest = [t.split(" ", 1)[1] if False else t for _, t in items[1:]]
            # drop the repeated verb phrase of later items: "... A stopped at the iteration limit in all ... and B in one run (...)"
            txt = first
            for (m, t) in items[1:]:
                tail = t[len(PROSE[m]):].strip()
                same_verb = first.split(" ", 1)[1].startswith(tail.split(" in ")[0]) if False else False
                verb = re.match(r"(stopped at the iteration limit|did not reach the tolerance)", tail).group(1)
                if first.find(verb) >= 0:
                    tail = tail[len(verb):].strip()
                txt += f" and {PROSE[m]} {tail}"
            clauses.append(f"those of {join_and(bases)} on {d['dataset']}, where {txt}")
    good = [m for m in M if m not in bad_methods]
    if not bad_methods:
        s1 = f"Every run of all methods reached the tolerance."
    else:
        s1 = (f"Every run of {join_and([PROSE[m] for m in good])} reached the tolerance; the only unsuccessful runs are "
              + "; and ".join(clauses) + ".") if good else "Unsuccessful runs: " + "; ".join(clauses) + "."
    s2 = (f" Thus ${nok}$ of the ${tot}$ runs (${len(M)}$ configurations, ${len(S.ds)}$ data sets, ${S.nr}$ starting points) were successful."
          if bad_methods else "")
    # replace words for numbers in s2 ($5$ etc. fine)
    s3 = (r" \crefrange{tab:logreg-itr}{tab:logreg-time} report the mean number of iterations, evaluations of $F$ and CPU time for each data set, "
          r"and \cref{tab:logreg-summary} aggregates them" + (f", separately for the {word(len(S.A))} a-sets and for {group_label(S.C)}," if S.A and S.C else "")
          + r" together with the number of evaluations of $F$. \cref{fig:logreg-profiles} shows performance profiles over all "
          + f"${len(S.ds) * S.nr}$ runs and \\cref{{fig:logreg-residual}} shows the residual histories of the first starting point on every data set.")
    return s1 + s2 + s3


def discussion(S):
    """Discussion in the condensed form of the report: one results paragraph, then the closing remarks."""
    A, C, M = S.A, S.C, METHODS
    sA = lambda m, k: S.mean(A, m, k)
    sC = lambda m, k: S.mean(C, m, k)
    nA = len(A)
    nRuns = len(S.ds) * S.nr
    rhp = PROSE[RHP]
    nok = int(sum(S.ok([d], RHP).sum() for d in S.ds))
    t = ""
    rhp_best_A = bool(A) and all(min(M, key=lambda m: sA(m, k)) == RHP for k in ("itr", "nf", "cpu"))
    t += (f"The results in \\crefrange{{tab:logreg-itr}}{{tab:logreg-time}}\n"
          + ("demonstrate the competitive performance of " if rhp_best_A or not A else "report the performance of ")
          + f"{rhp} on the regularized logistic regression problems considered.\n")
    if A:
        dec = {"itr": 1, "nf": 0, "cpu": 3}      # precision shown in the tables: values equal at that precision are ties

        def count(key):
            win = tie = 0
            for d in A:
                r = round(d["summary"][RHP][key], dec[key])
                o = min(round(d["summary"][m][key], dec[key]) for m in M if m != RHP)
                win += r < o
                tie += r == o
            return win, tie
        def w(kt):
            k, tt = kt
            if k == nA:
                return "on every data set"
            return (f"on {word(k)} data set" + ("" if k == 1 else "s") + (f", with a tie on {word(tt)} additional data set" + ("" if tt == 1 else "s") if tt else "")
                    if (tt or k != nA - 1) else f"on {word(k)}")
        t += (f"On the {word(nA)} Adult data sets, {rhp} achieves the lowest mean\n"
              f"iteration count {w(count('itr'))}, the lowest mean number\n"
              f"of function evaluations {w(count('nf'))}, and the lowest mean CPU\n"
              f"time {w(count('cpu'))}.\n")
    if A and C:
        r = sA(DF, "nf") / sA(RHP, "nf")
        if 1.8 <= r <= 2.3:
            cmp_ = f"requires approximately half the function evaluations of {PROSE[DF]}"
        elif r > 1:
            cmp_ = f"requires a factor of about ${ratio_fmt(r)}$ fewer function evaluations than {PROSE[DF]}"
        else:
            cmp_ = f"requires a factor of about ${ratio_fmt(1 / r)}$ more function evaluations than {PROSE[DF]}"
        t += (f"As summarized in \\cref{{tab:logreg-summary}}, {rhp} {cmp_}\n"
              f"when averaged over these {word(nA)} data sets.\n")
    if C:
        cn_ = C[0]["dataset"]
        w = min(M, key=lambda m: sC(m, "nf"))
        costs = {k: min(M, key=lambda m: sC(m, k)) for k in ("itr", "nf", "cpu")}
        if len(set(costs.values())) == 1:
            t += f"On {cn_}, {PROSE[w]} achieves the lowest computational\ncosts"
        else:
            t += f"On {cn_}, {PROSE[w]} needs the fewest evaluations of $F$"
        t += (f", while {rhp} also reaches the prescribed tolerance\nfrom all {word(S.nr)} starting points.\n" if S.ok(C, RHP).all()
              else f", while {rhp} reaches the prescribed tolerance in {int(S.ok(C, RHP).sum())} of {len(C) * S.nr} runs.\n")
    t += (f"Overall, {rhp} converges\nsuccessfully in all ${nRuns}$ runs.\n" if nok == nRuns
          else f"Overall, {rhp} converges\nsuccessfully in ${nok}$ of ${nRuns}$ runs.\n")
    t2 = ""
    if rhp_best_A:
        t2 += f"These findings support the effectiveness of {rhp},\nparticularly on the tested problems with $m\\gg n$.\n"
    t2 += "The conclusions are limited to the data sets,\nregularization weight, starting points and parameter\nsettings considered."
    rs = [r["restarts"] for d in S.ds for r in d["per_run"][RHP] if r.get("restarts") is not None]
    if rs and sum(rs) > 0:
        t2 += f" The restart safeguards were\nactivated in ${sum(1 for x in rs if x > 0)}$ of the ${len(rs)}$ runs of {rhp}."
    return t.rstrip() + "\n", t2.rstrip() + "\n"


# ------------------------------------------------------------------------------------------------------- driver
def build_text(S, template):
    T = template
    names = [d["dataset"] for d in S.ds]
    reg = S.env["reg"]
    tl = tex_num(S.tol)
    desc = []
    if S.A:
        desc.append(f"the {word(len(S.A))} LIBSVM test files {group_label(S.A)}" if len(S.A) > 1 else S.A[0]["dataset"])
    for d in S.C:
        desc.append(d["dataset"])
    cols, header, rows = data_table(S)
    rep = {
        "PLATFORM": "in Google Colab" if S.env.get("colab") else "on a local machine",
        "LAMBDA": tex_num(reg), "TOL": tl, "MAXIT": str(S.maxit),
        "NRUNS": word(S.nr), "NTOT": f"${len(S.ds) * S.nr}$", "NTOT_N": str(len(S.ds) * S.nr), "NMETH": word(len(METHODS)),
        "DATADESC": join_and(desc), "DATADESC_SHORT": (f"{word(len(S.A))} a-sets and {group_label(S.C)}" if S.A and S.C else join_and(names)),
        "NDS_WORD": word(len(S.ds)), "NAMES_SHORT": (f"{group_label(S.A)} and {group_label(S.C)}" if S.A and S.C else join_and(names)),
        "DATA_PAR": data_paragraph(S), "DATA_COLSPEC": cols, "DATA_HEADER": header, "DATA_ROWS": rows,
        "SYSTEM": system_sentence(S.env),
        "DPA_ELL": tex_num(S.set[DA]["ell"]), "DPA_U": tex_num(S.set[DA]["u"]), "DPB_ELL": tex_num(S.set[DB]["ell"]), "DPB_U": tex_num(S.set[DB]["u"]),
    }
    for k, v in S.set[RHP].items():
        rep["RHP_" + k] = tex_num(v)
    for k, v in S.set[DF].items():
        rep["DF_" + k] = tex_num(v)
    for k, v in S.set[DA].items():
        rep["DPA_" + k] = tex_num(v)
    for k, v in S.set[GN].items():
        rep["GN_" + k] = tex_num(v)
    rep["DPA_ELL"], rep["DPA_U"] = tex_num(S.set[DA]["ell"]), tex_num(S.set[DA]["u"])
    same = all(abs(S.set[RHP][k] - v) <= 1e-12 * max(1, abs(v)) for k, v in MANUSCRIPT_RHP.items() if k in S.set[RHP])
    rep["PARAMS_VS_SOLVERS"] = (r"The solver parameters used in this subsection are listed in \cref{tab:logreg-params}; they coincide with those of \cref{sec:solvers}." if same
                                else r"The solver parameters used in this subsection are listed in \cref{tab:logreg-params}; they are specific to these experiments and differ from those of \cref{sec:solvers}.")
    rep["SUCCESS_PAR"] = success_paragraph(S)
    rep["TABLES"] = make_tables(S)
    xl = getattr(S, "xl_a", 0)
    rep["XLIM_TEXT"] = (f"the horizontal axis ends at {xl} iterations for the a-sets and at {S.maxit} for {group_label(S.C)}" if S.A and S.C and xl else
                        f"the horizontal axis ends at {S.maxit} iterations")
    note = ""
    for d in S.C:
        stuck = [m for m in METHODS if S.T[d["dataset"]][m]["status"] != "converged"]
        if stuck:
            note = f" On {d['dataset']} {join_and([PROSE[m] for m in stuck])} " + ("terminates" if len(stuck) == 1 else "terminate") + " at the iteration limit without converging."
    rep["RESID_NOTE"] = note
    rep["DISCUSSION"], rep["DISCUSSION2"] = discussion(S)
    for k, v in rep.items():
        T = T.replace("@@" + k + "@@", v)
    left = re.findall(r"@@\w+@@", T)
    assert not left, f"unfilled placeholders: {left}"
    return T


STANDALONE = r"""\documentclass[12pt]{article}
\usepackage[a4paper,margin=2.5cm]{geometry}
\usepackage{amsmath,amssymb,amsthm,booktabs,graphicx}
\usepackage[numbers,sort&compress]{natbib}
\usepackage[colorlinks=true,linkcolor=blue,citecolor=blue,urlcolor=blue]{hyperref}
\usepackage[capitalise,noabbrev]{cleveref}
\newtheorem{proposition}{Proposition}\newtheorem{assumption}{Assumption}
\crefname{assumption}{Assumption}{Assumptions}
\usepackage{xcolor}
\usepackage{float} % Keep tables and figures at their source positions.
\begin{document}
% Stand-alone wrapper for compiling the report on its own (set this file as the main document).
% The cross-references to the manuscript (\cref{sec:solvers}, \eqref{eq:logreg}, ...) are undefined here.
\input{temp/report}

\begin{thebibliography}{9}
\bibitem[Chang and Lin(2011)]{changlin2011} C.-C. Chang and C.-J. Lin. LIBSVM: A library for support vector machines. \emph{ACM Trans. Intell. Syst. Technol.}, 2(3):1--27, 2011.
\bibitem[Dolan and Mor\'e(2002)]{dolan2002} E. D. Dolan and J. J. Mor\'e. Benchmarking optimization software with performance profiles. \emph{Math. Program.}, 91(2):201--213, 2002.
\bibitem[La Cruz et~al.(2006)]{lacruz2006} W. La Cruz, J. M. Mart\'inez and M. Raydan. Spectral residual method without gradient information for solving large-scale nonlinear systems of equations. \emph{Math. Comp.}, 75(255):1429--1448, 2006.
\bibitem[Li and Fukushima(1999)]{lifukushima1999} D.-H. Li and M. Fukushima. A globally and superlinearly convergent Gauss--Newton-based BFGS method for symmetric nonlinear equations. \emph{SIAM J. Numer. Anal.}, 37(1):152--172, 1999.
\bibitem[Mohammad and Awwal(2022)]{mohammadawwal2022} H. Mohammad and A. M. Awwal. Globally convergent diagonal Polak--Ribi\`ere--Polyak like algorithm for nonlinear equations. \emph{Numer. Algorithms}, 91(4):1441--1460, 2022.
\bibitem{inertialpaper} J. Jian, J. Yin, C. Tang and D. Han. A family of inertial derivative-free projection methods for constrained nonlinear pseudo-monotone equations with applications. \emph{Comput. Appl. Math.}, 41:309, 2022.
\end{thebibliography}
\end{document}
"""


def make_report(results_path="results.json", traces_path=None, outdir="temp", template="report_template.tex", template_text=None):
    R = json.load(open(results_path))
    traces_path = traces_path or os.path.splitext(results_path)[0] + "_traces.json"
    T = json.load(open(traces_path))
    S = Stats(R, T)
    os.makedirs(outdir, exist_ok=True)
    global PLOT
    PLOT = PLOT_NAMES[NAME_STYLE]
    make_figures(S, outdir)               # sets S.xl_a
    tpl = template_text if template_text is not None else open(template).read()
    text = apply_names(build_text(S, tpl), NAME_STYLE)
    with open(os.path.join(outdir, "report.tex"), "w") as fh:
        fh.write(text)
    with open(os.path.join(outdir, "report_standalone.tex"), "w") as fh:
        fh.write(apply_names(STANDALONE, NAME_STYLE))
    return text


def zip_temp(outdir="temp"):
    """temp.zip containing the folder temp/ (extract it and replace the Overleaf folder 'temp')."""
    parent = os.path.dirname(os.path.abspath(outdir))
    base = os.path.basename(os.path.normpath(outdir))
    z = shutil.make_archive(os.path.join(parent, base), "zip", root_dir=parent, base_dir=base)
    return z

In [ ]:
REPORT_TEMPLATE = r'''\subsection{Numerical experiments: regularized logistic regression}\label{sec:logreg-experiment}

This section assesses the proposed restarted hybridplus method (RHP) on a large-scale, strongly convex test problem that arises in machine learning, and compares it with three derivative-free solvers of nonlinear equations: DF-SANE \cite{lacruz2006}, DPRP$^{+}$ \cite{mohammadawwal2022} and GN-BFGS \cite{lifukushima1999}. We solve the gradient equation $F=\nabla\phi=0$ of the $\ell_2$-regularized logistic regression problem \eqref{eq:logreg} with $\lambda=@@LAMBDA@@$. The data matrix $A\in\mathbb{R}^{m\times n}$ is the one loaded from the LIBSVM file, with the labels mapped to $\{-1,+1\}$.

The experiments of this subsection differ in kind from those reported above for systems of nonlinear equations. They concern a single structured problem class, for which $F$ is the gradient of a strongly convex function and its Jacobian is symmetric positive definite. They were not run with the MATLAB framework of \cref{sec:solvers}; they were run with a separate Python~3 implementation (NumPy, SciPy and scikit-learn) @@PLATFORM@@.

@@DATA_PAR@@

\begin{table}[H]
\centering
\caption{Data sets: number of observations $m$ and number of variables $n$.}
\label{tab:logreg-data}
\small
\begin{tabular}{@@DATA_COLSPEC@@}
\toprule
@@DATA_HEADER@@ \\
\midrule
@@DATA_ROWS@@
\bottomrule
\end{tabular}
\end{table}

For each data set every method is run from the same @@NRUNS@@ random starting points $x_0=4(\mathrm{rand}(n,1)-0.5)$, i.e.\ uniformly distributed in $[-2,2]^{n}$. The points are drawn with a seeded NumPy generator (re-seeded for each data set, so that data sets with equal $n$ share the same starting points). A run is declared successful if $\|F(x_k)\|\le@@TOL@@$ is reached within $@@MAXIT@@$ iterations, and unsuccessful otherwise. All reported figures are means over the @@NRUNS@@ runs.

The configurations are summarized in \cref{tab:logreg-params}; they are specific to these experiments and differ from those of \cref{sec:solvers}. {\color{black} RHP uses identity scaling $B_k=I$ and the finite-difference rule for the initial trial step of the line search, with parameters selected through preliminary runs.} For DPRP$^{+}$ the lower and upper safeguards $(\ell,u)$ of the diagonal scaling matrix and the line-search parameter $\sigma$ are not specified in \cite{mohammadawwal2022}; we therefore report two settings, DPRP$^{+}_{\mathrm A}$ with $(\ell,u)=(@@DPA_ELL@@,@@DPA_U@@)$ and DPRP$^{+}_{\mathrm B}$ with $(\ell,u)=(@@DPB_ELL@@,@@DPB_U@@)$, the latter having been selected after a short trial on a single starting point of a1a.t and of colon-cancer and being therefore favourable to DPRP$^{+}$. {\color{black} The sequence $\tau_k=e^{-(k+1)^{k+1}}$ of \cite{mohammadawwal2022} underflows to zero in double precision for $k\ge4$, so that the line search of DPRP$^{+}$ is in practice monotone after the first few iterations.} GN-BFGS is implemented with a dense $n\times n$ matrix $B_k$ (initialized to $I$) and a dense linear solve per iteration.

\begin{table}[H]
\centering
\caption{Algorithm configurations used in the experiments.}
\label{tab:logreg-params}
\small
\setlength{\tabcolsep}{4pt}
\begin{tabular}{lp{10.4cm}}
\toprule
Method & Parameters \\
\midrule
RHP & finite-difference initial step; identity scaling $B_k=I$; $t=@@RHP_t@@$, $\rho=@@RHP_rho@@$, $\sigma=@@RHP_sigma@@$, $\ell_{\max}=@@RHP_lmax@@$, $\eta_k=(k+1)^{-2}$, $\ell_{\min}=@@RHP_lmin@@$, $\nu=@@RHP_nu@@$ \\
DF-SANE \cite{lacruz2006} & $M=@@DF_M@@$, $\gamma=@@DF_gamma@@$, $\tau_{\min}=@@DF_tau_min@@$, $\tau_{\max}=@@DF_tau_max@@$, $\sigma_{\min}=@@DF_sigma_eps@@$, $\sigma_{\max}=10^{10}$, $\sigma_0=@@DF_sigma0@@$, $\eta_k=\|F(x_0)\|/(1+k)^2$ \\
DPRP$^{+}_{\mathrm A}$, DPRP$^{+}_{\mathrm B}$ \cite{mohammadawwal2022} & $\theta=@@DPA_theta@@$, $\rho=@@DPA_rho@@$, $\tau_k=e^{-(k+1)^{k+1}}$, $\sigma=@@DPA_sigma@@$; $(\ell,u)=(@@DPA_ELL@@,@@DPA_U@@)$ (A) and $(@@DPB_ELL@@,@@DPB_U@@)$ (B) \\
GN-BFGS \cite{lifukushima1999} & $r=@@GN_r@@$, $\rho=@@GN_rho@@$, $\sigma_1=\sigma_2=@@GN_sigma1@@$, $\lambda_{-1}=@@GN_lam_prev@@$, $\varepsilon_k=(k+1)^{-2}$, $B_0=I$ \\
\bottomrule
\end{tabular}
\end{table}

We record (i) the number of iterations, (ii) the number of evaluations of $F$ and (iii) the CPU time in seconds (process CPU time, data loading excluded). In addition to means, we summarize the run-level behaviour with Dolan--Mor\'e performance profiles \cite{dolan2002} computed over all $@@NTOT_N@@$ runs, in which an unsuccessful run is assigned an infinite ratio.

@@SYSTEM@@ Timings in the millisecond range are subject to measurement noise.

@@SUCCESS_PAR@@

@@TABLES@@
@@DISCUSSION@@
\begin{figure}[H]
\centering
\includegraphics[width=\textwidth]{temp/fig_profiles.pdf}
\caption{Dolan--Mor\'e performance profiles over all $@@NTOT_N@@$ runs (@@DATADESC_SHORT@@, @@NRUNS@@ starting points each) for the @@NMETH@@ methods, for (a) the number of evaluations of $F$ and (b) the CPU time. For each method the curve gives the fraction of runs in which its cost is at most $\tau$ times the smallest cost attained by any method on that run; unsuccessful runs have an infinite ratio. DPRP$^{+}_{\mathrm A}$ and DPRP$^{+}_{\mathrm B}$ denote the settings $(\ell,u)=(@@DPA_ELL@@,@@DPA_U@@)$ and $(@@DPB_ELL@@,@@DPB_U@@)$.}
\label{fig:logreg-profiles}
\end{figure}

\begin{figure}[H]
\centering
\includegraphics[width=\textwidth]{temp/fig_residual.pdf}
\caption{Residual norm $\|F(x_k)\|$ against the iteration counter for the first of the @@NRUNS@@ starting points on each of the @@NDS_WORD@@ data sets (@@NAMES_SHORT@@; @@XLIM_TEXT@@). The dotted line marks the tolerance $@@TOL@@$.@@RESID_NOTE@@ Single runs are shown for illustration; run-to-run variation is quantified in \crefrange{tab:logreg-itr}{tab:logreg-time} and \cref{fig:logreg-profiles}.}
\label{fig:logreg-residual}
\end{figure}

@@DISCUSSION2@@
'''

In [ ]:
# Regenerates the report from results.json, builds the temp/ folder and downloads temp.zip (Colab) -- or leaves it next to the notebook.
text = make_report("results.json", "results_traces.json", "temp", template_text=REPORT_TEMPLATE)
zip_path = zip_temp("temp")
print("temp/ contains:", sorted(os.listdir("temp")))
print("archive:", zip_path)
try:
    from google.colab import files
    files.download(zip_path)          # the browser asks where to save temp.zip
except ImportError:
    print("Not running on Colab: temp.zip was written next to the notebook.")

temp/ contains: ['fig_profiles.pdf', 'fig_residual.pdf', 'report.tex', 'report_standalone.tex']
archive: /content/temp.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Known differences and limits
* DF-SANE follows the 2006 paper (variant "paper"); the SciPy-compatible variant is validated against SciPy, and the paper variant is the one run.
* DPRP+: `sigma`, `l`, `u` are not specified in the paper; results depend strongly on `(l, u)` (see the two settings).
* GN-BFGS stores a dense `B_k` (n = 122/123 on the a-sets, 2000 on colon-cancer) and factorises it every iteration, O(n^3) work that NF does not count; CPU time is the more honest cost measure for it (about 30 s per colon-cancer run) and is part of the comparison.
* DPRP+ signs: eq. (4), (7), the line search (12) and Algorithm 1 Step 4 all use `x + lambda d`, `d = -Q F + beta d_prev` in the PDF used here; no sign inconsistency was found.
  Its NF counts every `g` evaluation including the probe `g(x_k + lambda_{k-1} g_k)` and `g(x_k + delta_k)`.
* The RHP restart tests (`d = 0`, `w = 0`) are exact comparisons with zero, as in the paper's algorithm; in floating
  point they never fire here (`restarts = 0` in every run), so this experiment does not exercise the restart rule.
* Only identity-scaled RHP (`B_k = I`) is implemented, not the diagonally scaled version (eq. (13)).